# Aircraft Classification Demo — Initial Five-Class Prototype

This notebook presents the **initial and simplified version** of the aircraft classification project.

It corresponds to the first stage of the work: a **small five-class prototype** designed to validate the core approach before developing the larger FGVC-Aircraft models used later for manufacturer and family classification.

## Position within the broader project

This notebook should be read as the **starting point** of the project:
- a smaller and easier classification problem,
- a lighter dataset setup,
- a simpler training workflow,
- a first proof that the end-to-end pipeline works.

The larger FGVC-Aircraft notebook developed afterwards extends the same logic to a much more demanding setting:
- more classes,
- more data,
- more structured preprocessing,
- more thorough evaluation.

## Why this notebook matters

This first prototype illustrates a sound engineering progression:
1. start with a limited and controlled experiment,
2. validate the training and inference pipeline,
3. export a working model,
4. use this first success as the basis for a more advanced version.

## What this notebook contains

The workflow includes:
- environment setup,
- dataset loading,
- FastAI data pipeline construction,
- model definition,
- model training,
- model export,
- basic inference checks.

## Reading guideline

The code logic has been preserved on purpose.  
This cleaned version is meant to make the notebook easier to read and more suitable for a technical portfolio or engineering school application, while clearly showing that this notebook corresponds to the **initial, simpler stage** of the overall project.


## Imports and environment setup

This section imports the Python libraries used throughout the notebook and prepares the execution environment for the initial prototype.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This cell belongs to the initial five-class prototype.
# It documents the first and simpler stage of the aircraft-classification
# project, whose purpose was to validate the end-to-end workflow before
# scaling up to the broader FGVC-Aircraft models.

# Import the libraries required for data handling, model training, and inference.
from fastcore.all import *
from fastai.vision.all import *
from fastai.vision.widgets import *
from fastdownload import download_url

## Notebook step

This section contains part of the initial five-class workflow and documents how the project started from a smaller and simpler proof of concept.

In [ ]:
# -----------------------------------------------------------------------------
# NOTEBOOK STEP
# -----------------------------------------------------------------------------
# This cell belongs to the initial five-class prototype.
# It documents the first and simpler stage of the aircraft-classification
# project, whose purpose was to validate the end-to-end workflow before
# scaling up to the broader FGVC-Aircraft models.

pip install ddgs

## Imports and environment setup

This section imports the Python libraries used throughout the notebook and prepares the execution environment for the initial prototype.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This cell belongs to the initial five-class prototype.
# It documents the first and simpler stage of the aircraft-classification
# project, whose purpose was to validate the end-to-end workflow before
# scaling up to the broader FGVC-Aircraft models.

# Import the libraries required for data handling, model training, and inference.
from fastcore.all import *
from fastai.vision.all import *
from fastai.vision.widgets import *
from fastdownload import download_url

## Imports and environment setup

This section imports the Python libraries used throughout the notebook and prepares the execution environment for the initial prototype.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This cell belongs to the initial five-class prototype.
# It documents the first and simpler stage of the aircraft-classification
# project, whose purpose was to validate the end-to-end workflow before
# scaling up to the broader FGVC-Aircraft models.

# Import the libraries required for data handling, model training, and inference.
from ddgs import DDGS
from pathlib import Path
from time import sleep
from tqdm import tqdm

from fastai.vision.all import *
from fastai.vision.widgets import *
from ddgs import DDGS

# Define the helper function `search_images` used later in the notebook.
def search_images(query, max_results=30):
    with DDGS(timeout=30) as ddgs:
        results = ddgs.images(
            query,
            max_results=max_results,
            safesearch="off",
            region="wt-wt"
        )
    return [r["image"] for r in results if "image" in r]

avions = ['Dassault Rafale', 'Piper Cub', 'Boeing 737', 'McDonnell Douglas C-17', 'Citation XL']
# Define a filesystem path used later in the notebook.
# Store the main working path in a dedicated variable.
path = Path('avions')

for b in tqdm(avions):
    dest = path / b
    dest.mkdir(exist_ok=True, parents=True)

    download_images(dest, urls=search_images(f'photo {b}'))
    sleep(10)

    download_images(dest, urls=search_images(f'photo {b} from above'))
    sleep(10)

    download_images(dest, urls=search_images(f'photo {b} from below'))
    sleep(10)

    resize_images(dest, max_size=400)

## Imports and environment setup

This section imports the Python libraries used throughout the notebook and prepares the execution environment for the initial prototype.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This cell belongs to the initial five-class prototype.
# It documents the first and simpler stage of the aircraft-classification
# project, whose purpose was to validate the end-to-end workflow before
# scaling up to the broader FGVC-Aircraft models.

# Import the libraries required for data handling, model training, and inference.
from fastai.vision.utils import verify_images

# Collect the image files that will feed the data pipeline.
fns = get_image_files(path)
failed = verify_images(fns)
failed

## Notebook step

This section contains part of the initial five-class workflow and documents how the project started from a smaller and simpler proof of concept.

In [ ]:
# -----------------------------------------------------------------------------
# NOTEBOOK STEP
# -----------------------------------------------------------------------------
# This cell belongs to the initial five-class prototype.
# It documents the first and simpler stage of the aircraft-classification
# project, whose purpose was to validate the end-to-end workflow before
# scaling up to the broader FGVC-Aircraft models.

failed.map(Path.unlink)

## Dataset paths and file discovery

This section defines the dataset paths and inspects the file structure used by the initial five-class experiment.

In [ ]:
# -----------------------------------------------------------------------------
# DATASET PATHS AND FILE DISCOVERY
# -----------------------------------------------------------------------------
# This cell belongs to the initial five-class prototype.
# It documents the first and simpler stage of the aircraft-classification
# project, whose purpose was to validate the end-to-end workflow before
# scaling up to the broader FGVC-Aircraft models.

# Build the FastAI DataBlock that defines how images and labels are loaded.
# Store the DataLoaders object in a dedicated variable.
dls = DataBlock(
    blocks = (ImageBlock, CategoryBlock),
# Collect the image files that will feed the data pipeline.
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed = 18),
    get_y=parent_label,
    item_tfms=[Resize(256, method='squish')]
# Build the FastAI DataLoaders object used to serve mini-batches during training.
).dataloaders(path, bs=32)

## Dataset paths and file discovery

This section defines the dataset paths and inspects the file structure used by the initial five-class experiment.

In [ ]:
# -----------------------------------------------------------------------------
# DATASET PATHS AND FILE DISCOVERY
# -----------------------------------------------------------------------------
# This cell belongs to the initial five-class prototype.
# It documents the first and simpler stage of the aircraft-classification
# project, whose purpose was to validate the end-to-end workflow before
# scaling up to the broader FGVC-Aircraft models.

# Build the FastAI DataLoaders object used to serve mini-batches during training.
# Store the DataLoaders object in a dedicated variable.
dls = ImageDataLoaders.from_folder(path, valid_pct=0.2, seed=42, item_tfms=Resize(224))
# Create the learner object that combines the model, the data, and the training configuration.
# Use a ResNet backbone for image classification in this first prototype.
# Create the learner used for both training and inference.
learn = vision_learner(dls, resnet34, metrics=accuracy)
# Fine-tune the model: first train the head, then unfreeze and continue training.
learn.fine_tune(5)

## Imports and environment setup

This section imports the Python libraries used throughout the notebook and prepares the execution environment for the initial prototype.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This cell belongs to the initial five-class prototype.
# It documents the first and simpler stage of the aircraft-classification
# project, whose purpose was to validate the end-to-end workflow before
# scaling up to the broader FGVC-Aircraft models.

# Import the libraries required for data handling, model training, and inference.
from fastai.vision.all import ClassificationInterpretation

interp = ClassificationInterpretation.from_learner(learn)

interp.plot_confusion_matrix(normalize = True)

## Model export

This section saves the trained model artifact so that it can be reused later for inference or deployment.

In [ ]:
# -----------------------------------------------------------------------------
# MODEL EXPORT
# -----------------------------------------------------------------------------
# This cell belongs to the initial five-class prototype.
# It documents the first and simpler stage of the aircraft-classification
# project, whose purpose was to validate the end-to-end workflow before
# scaling up to the broader FGVC-Aircraft models.

# Export the trained model so it can be reloaded later without retraining.
learn.export('airplane_recognition_model.pkl')

## Imports and environment setup

This section imports the Python libraries used throughout the notebook and prepares the execution environment for the initial prototype.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This cell belongs to the initial five-class prototype.
# It documents the first and simpler stage of the aircraft-classification
# project, whose purpose was to validate the end-to-end workflow before
# scaling up to the broader FGVC-Aircraft models.

!pip -q install torchview graphviz

# Import the libraries required for data handling, model training, and inference.
from torchview import draw_graph
import torch

model = learn.model.eval()
graph = draw_graph(model, input_size=(1,3,224,224), expand_nested=True)
graph.visual_graph.render(
    filename="model_graph",
    format="png",
    cleanup=True
)

graph.visual_graph